In [33]:
import pandas as pd
from fredapi import Fred
from settings import fred_api_key

In [34]:
# Import df from csv
raw_df = pd.read_csv('raw_data.csv')

In [44]:
## FROM RAW DATA TO df_y

# Load and clean columns
df = raw_df.iloc[1:].copy()
df.columns = df.columns.str.strip()

# Melt to long format: 'Capital IQ Information' holds info, others are tickers
df_long = df.melt(id_vars=['Capital IQ Information'], var_name='Ticker', value_name='Value')

# Clean numeric values
df_long['Value'] = df_long['Value'].str.replace(',', '').str.strip()
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

# Filter only median revenue consensus estimates
mask = df_long['Capital IQ Information'].str.contains('Revenue Median Consensus Estimate', na=False)
df_median = df_long[mask].copy()

# Extract forecast year
df_median['Year'] = df_median['Capital IQ Information'].str.extract(r'CY (\d{4})')[0].astype(int)

# Drop unnecessary column and rename for clarity
df_median = df_median.rename(columns={'Value': 'RevenueForecast'})

# Keep only necessary columns and drop missing values
df_y = df_median[['Ticker', 'Year', 'RevenueForecast']].dropna()

print(df_y.head(10))


   Ticker  Year  RevenueForecast
5    AAPL  2025          406.361
6    AAPL  2026          431.829
7    AAPL  2027          459.682
8    AAPL  2028          455.481
9    AAPL  2029          487.125
39    HPE  2025           32.519
40    HPE  2026           34.134
41    HPE  2027           35.133
42    HPE  2028           35.839
43    HPE  2029           35.881


In [47]:
import pandas as pd
import numpy as np

# Load raw data
raw_df = pd.read_csv('raw_data.csv')
raw_df.columns = raw_df.columns.str.strip()

# Melt into long format
tickers = raw_df.columns.drop('Capital IQ Information')
df_long = raw_df.melt(id_vars='Capital IQ Information', value_vars=tickers,
                      var_name='Ticker', value_name='Value')

# Clean strings
df_long['Capital IQ Information'] = df_long['Capital IQ Information'].str.strip()
df_long['Value'] = df_long['Value'].astype(str).str.strip()


# Function to convert financial string values, including negatives in parentheses
def convert_financial_value(val):
    if val in ['n/a', 'NA', '', None, np.nan]:
        return np.nan
    val = val.replace(',', '')
    if val.startswith('(') and val.endswith(')'):
        val = '-' + val[1:-1]
    try:
        return float(val)
    except:
        return np.nan


df_long['Value'] = df_long['Value'].apply(convert_financial_value)

# Filter only historical EBITDA and Revenue rows
mask_hist = df_long['Capital IQ Information'].str.contains('Last FY')
df_hist = df_long[mask_hist].copy()

# Extract Year information:
# Suppose your valuation date is CY 2024, so:
# Last FY -> 2023
# Last FY - 1 -> 2022
# Last FY - 2 -> 2021
# ... adjust accordingly

valuation_year = 2025


def extract_hist_year(row):
    if 'Last FY -' in row:
        n = int(row.split('Last FY -')[-1].strip())
        return valuation_year - 1 - n
    elif 'Last FY' in row:
        return valuation_year - 1
    else:
        return np.nan


df_hist['Year'] = df_hist['Capital IQ Information'].apply(extract_hist_year)

# Extract FeatureType (EBITDA or Revenue)
df_hist['FeatureType'] = df_hist['Capital IQ Information'].str.extract(r'(EBITDA|Revenue)')

# Pivot so each feature is a column
df_features_hist = df_hist.pivot_table(index=['Ticker', 'Year'],
                                       columns='FeatureType',
                                       values='Value').reset_index()

# Clean column names
df_features_hist.columns.name = None

print(df_features_hist.head(30))


   Ticker  Year   EBITDA  Revenue
0    AAPL  2019   76.477  260.174
1    AAPL  2020   77.344  274.515
2    AAPL  2021  120.233  365.817
3    AAPL  2022  130.541  394.328
4    AAPL  2023  125.820  383.285
5    AAPL  2024  134.661  391.035
6    CSCO  2019   16.438   51.904
7    CSCO  2020   15.909   49.301
8    CSCO  2021   15.581   49.818
9    CSCO  2022   15.932   51.557
10   CSCO  2023   17.288   56.998
11   CSCO  2024   15.477   53.803
12     GD  2019    5.399   39.350
13     GD  2020    5.011   37.925
14     GD  2021    5.053   38.469
15     GD  2022    4.797   39.407
16     GD  2023    4.853   42.272
17     GD  2024    5.440   47.716
18    HPE  2019    5.071   29.135
19    HPE  2020    4.624   26.982
20    HPE  2021    3.569   27.290
21    HPE  2022    4.224   28.013
22    HPE  2023    4.511   28.588
23    HPE  2024    4.390   29.459
24   QBTS  2020  -29.000    5.000
25   QBTS  2021  -36.000    6.000
26   QBTS  2022  -58.000    7.000
27   QBTS  2023  -79.000    9.000
28   QBTS  202

In [30]:
# BUILD RISK FREE RATE DF
# --- Set your FRED API key here ---
fred = Fred(api_key=fred_api_key)
gs10 = fred.get_series('GS10', observation_start='2020-01-01', observation_end='2029-12-31')

# Convert to DataFrame
df_gs10 = pd.DataFrame(gs10)
df_gs10.index = pd.to_datetime(df_gs10.index)
df_gs10.columns = ['Risk_Free_Rate']

# Resample to annual average yield
df_risk_free = df_gs10.resample('YE').mean().reset_index()

# Extract Year
df_risk_free['Year'] = df_risk_free['index'].dt.year

# Keep only needed columns and sort
df_risk_free = df_risk_free[['Year', 'Risk_Free_Rate']].sort_values('Year').reset_index(drop=True)

# --- INFLATION DF ----
cpi = fred.get_series('CPIAUCSL', observation_start='2019-12-31', observation_end='2029-12-31')
# Convert to DataFrame
df_cpi = pd.DataFrame(cpi)
df_cpi.index = pd.to_datetime(df_cpi.index)
df_cpi.columns = ['CPI']

# Resample to annual average CPI
df_cpi_annual = df_cpi.resample('YE').mean().reset_index()

# Extract Year
df_cpi_annual['Year'] = df_cpi_annual['index'].dt.year

# Calculate YoY % change in CPI as Inflation Rate
df_cpi_annual['Inflation'] = df_cpi_annual['CPI'].pct_change() * 100

# Keep only Year and Inflation columns, drop first NaN row
df_cpi_final = df_cpi_annual[['Year', 'Inflation']].dropna().reset_index(drop=True)

# ----- GDP DF -----
# Fetch GDP data from FRED
gdp = fred.get_series('GDP', observation_start='2019-12-31', observation_end='2029-12-31')

# Convert to DataFrame
df_gdp = pd.DataFrame(gdp)
df_gdp.index = pd.to_datetime(df_gdp.index)
df_gdp.columns = ['GDP']

# Resample to annual average GDP
df_gdp_annual = df_gdp.resample('YE').mean().reset_index()

# Extract Year
df_gdp_annual['Year'] = df_gdp_annual['index'].dt.year

# Calculate YoY % change in GDP as GDP Growth Rate
df_gdp_annual['GDP_Growth'] = df_gdp_annual['GDP'].pct_change() * 100

# Keep only Year and GDP_Growth columns, drop first NaN row
df_gdp_final = df_gdp_annual[['Year', 'GDP_Growth']].dropna().reset_index(drop=True)

# MERGE DFs

# Merge sequentially on 'Year'
df_macro = df_risk_free.merge(df_cpi_final, on='Year', how='outer') \
    .merge(df_gdp_final, on='Year', how='outer')

# Sort by Year and reset index
df_macro = df_macro.sort_values('Year').reset_index(drop=True)

# Display results
print(df_macro)

   Year  Risk_Free_Rate  Inflation  GDP_Growth
0  2020        0.894167   0.087287   -2.640343
1  2021        1.442500   4.679118   10.897511
2  2022        2.951667   7.992644    9.820975
3  2023        3.957500   4.127717    6.589857
4  2024        4.208333   2.951606    5.281902
5  2025        4.406667   2.055587    2.662874


In [58]:
# Assuming df_features_hist and df_macro have 'Year' column as int or compatible dtype

df_X = pd.merge(df_features_hist, df_macro, on='Year', how='left')

print(df_X.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Ticker          34 non-null     object 
 1   Year            34 non-null     int64  
 2   EBITDA          34 non-null     float64
 3   Revenue         32 non-null     float64
 4   Risk_Free_Rate  30 non-null     float64
 5   Inflation       30 non-null     float64
 6   GDP_Growth      30 non-null     float64
dtypes: float64(5), int64(1), object(1)
memory usage: 2.0+ KB
None


In [59]:
# Define baseline values (replace with actual known values)
baseline_inflation_2019 = 1.8  # example: 1.8%
baseline_gdp_growth_2019 = 2.3  # example: 2.3%
baseline_risk_free_rate_2019 = 1.9  # example: 1.9%

# Fill missing macro values with these baseline values
df_X['Inflation'] = df_X['Inflation'].fillna(baseline_inflation_2019)
df_X['GDP_Growth'] = df_X['GDP_Growth'].fillna(baseline_gdp_growth_2019)
df_X['Risk_Free_Rate'] = df_X['Risk_Free_Rate'].fillna(baseline_risk_free_rate_2019)


In [60]:
print(df_X.head(20))
print(df_X.info())

   Ticker  Year   EBITDA  Revenue  Risk_Free_Rate  Inflation  GDP_Growth
0    AAPL  2019   76.477  260.174        1.900000   1.800000    2.300000
1    AAPL  2020   77.344  274.515        0.894167   0.087287   -2.640343
2    AAPL  2021  120.233  365.817        1.442500   4.679118   10.897511
3    AAPL  2022  130.541  394.328        2.951667   7.992644    9.820975
4    AAPL  2023  125.820  383.285        3.957500   4.127717    6.589857
5    AAPL  2024  134.661  391.035        4.208333   2.951606    5.281902
6    CSCO  2019   16.438   51.904        1.900000   1.800000    2.300000
7    CSCO  2020   15.909   49.301        0.894167   0.087287   -2.640343
8    CSCO  2021   15.581   49.818        1.442500   4.679118   10.897511
9    CSCO  2022   15.932   51.557        2.951667   7.992644    9.820975
10   CSCO  2023   17.288   56.998        3.957500   4.127717    6.589857
11   CSCO  2024   15.477   53.803        4.208333   2.951606    5.281902
12     GD  2019    5.399   39.350        1.900000  